# (7065) Fredschaaf — первая редукция серии

Этот notebook предназначен для первого контролируемого результата: таблицы положений слабого астероида в пределах одной серии и положения на её центральный момент. Эфемерида здесь нужна только как начальная подсказка; движение определяется по самим кадрам.

Запускать из корня проекта через `./run_jupyter.sh`; kernel — **Python (Fredschaaf astrometry)**. Сначала надо пройти все ячейки до поиска астероида. Нельзя считать точность существующего WCS окончательной астрометрией Gaia.

In [ ]:
from pathlib import Path
from functools import lru_cache
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.time import Time, TimeDelta
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from photutils.centroids import centroid_2dg
from photutils.detection import DAOStarFinder
from scipy.ndimage import shift as image_shift
from scipy.spatial import cKDTree

ROOT = Path.cwd(); DATA_ROOT = ROOT / 'Fredschaaf'; OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
# Начать с WCS-решенной серии R. Не объединять R и H-alpha.
DATE, FILTER, MAX_FRAMES = '20250903', 'R', None
SATURATION, EDGE = 65000, 30
assert DATA_ROOT.exists(), f'Нет данных в {DATA_ROOT}'

## 1. Инвентарь серии

Проверяем однородность кадров и время. `end_minus_start_s` сопоставляется с `EXPTIME`: прежде чем стремиться к миллисекундной точности, надо знать физический смысл обоих полей заголовка.

In [ ]:
def mid_time(h):
    return Time(h['DATE-OBS'], format='isot', scale='utc') + TimeDelta(float(h['EXPTIME'])/2, format='sec')

def frame_record(path):
    with fits.open(path, memmap=False) as f:
        h, data = f[0].header, f[0].data.astype(float)
    _, bkg, rms = sigma_clipped_stats(data, sigma=3, maxiters=5)
    start = Time(h['DATE-OBS'], format='isot', scale='utc')
    end = Time(h['DATE-END'], format='isot', scale='utc') if 'DATE-END' in h else None
    try: has_wcs = WCS(h).has_celestial
    except Exception: has_wcs = False
    return dict(path=str(path), filename=path.name, filter=h.get('FILTER'), mid_utc=mid_time(h).isot,
        jd_utc=mid_time(h).jd, exptime_s=float(h['EXPTIME']), gain=h.get('GAIN'), has_wcs=has_wcs,
        end_minus_start_s=(end-start).to_value(u.s) if end else np.nan, background_adu=bkg,
        background_rms_adu=rms, n_saturated=np.count_nonzero(data >= SATURATION))

paths = sorted((DATA_ROOT / DATE).glob('*.fits'))
paths = [p for p in paths if f'_{FILTER}_' in p.name]
if MAX_FRAMES is not None: paths = paths[:MAX_FRAMES]
assert paths, 'Нет кадров с выбранными DATE и FILTER'
frames = pd.DataFrame(frame_record(p) for p in paths).sort_values('jd_utc').reset_index(drop=True)
frames.to_csv(OUTPUT / f'frames_{DATE}_{FILTER}.csv', index=False)
display(frames)
display(frames[['exptime_s','end_minus_start_s','gain','has_wcs','background_rms_adu','n_saturated']].describe())
print(f"{len(frames)} кадров, длительность {(frames.jd_utc.iloc[-1]-frames.jd_utc.iloc[0])*1440:.2f} min")

## 2. Контроль одного кадра

Осмотрите поле, насыщения, фокус и трекинг. Сетка RA/Dec проверяет WCS лишь качественно. В окончательном конвейере геометрию следует заново определить по Gaia и переносить её ошибку в ковариацию астероида.

In [ ]:
@lru_cache
def read_frame(path):
    with fits.open(path, memmap=False) as f: return f[0].data.astype(float), f[0].header.copy()

reference_index = 0
reference_data, reference_header = read_frame(frames.path.iloc[reference_index])
reference_wcs = WCS(reference_header)
_, bkg, rms = sigma_clipped_stats(reference_data, sigma=3)
fig, ax = plt.subplots(figsize=(11,7), subplot_kw={'projection':reference_wcs} if reference_wcs.has_celestial else None)
ax.imshow(reference_data, origin='lower', cmap='gray', vmin=bkg-rms, vmax=bkg+8*rms)
if reference_wcs.has_celestial:
    ax.coords.grid(color='white', ls=':', alpha=.5); ax.set_xlabel('RA'); ax.set_ylabel('Dec')
    scale = np.mean(proj_plane_pixel_scales(reference_wcs))*u.deg/u.pix
    print(f'Приблизительный масштаб WCS: {scale.to(u.arcsec/u.pix):.4f}')
else: ax.set_xlabel('x, pix'); ax.set_ylabel('y, pix')
ax.set_title(frames.filename.iloc[reference_index]); plt.show()

## 3. Сдвиги по звездам и звездный стек

Это диагностическая, трансляционная модель регистрации. Большие скачки сдвига, малое число совпадений или размытые звезды на стеке — повод отбросить кадр и разобраться, а не усреднять его. На следующей итерации её заменит общая модель дисторсии/ePSF.

In [ ]:
def find_stars(data, fwhm=4., threshold_sigma=8.):
    _, bkg, rms = sigma_clipped_stats(data, sigma=3, maxiters=5)
    table = DAOStarFinder(fwhm=fwhm, threshold=threshold_sigma*rms, exclude_border=True)(data-bkg)
    if table is None: return np.empty((0,2))
    good = ((table['xcentroid']>EDGE)&(table['xcentroid']<data.shape[1]-EDGE)&
            (table['ycentroid']>EDGE)&(table['ycentroid']<data.shape[0]-EDGE))
    return np.c_[np.asarray(table['x_centroid'][good]), np.asarray(table['y_centroid'][good])]

def shift_to_reference(ref_xy, xy, initial_radius=18.):
    if min(len(ref_xy), len(xy)) < 4: return np.nan, np.nan, 0
    tree = cKDTree(ref_xy); distance, ind = tree.query(xy, distance_upper_bound=initial_radius)
    good = np.isfinite(distance) & (distance < initial_radius)
    if good.sum() < 4: return np.nan, np.nan, int(good.sum())
    dx, dy = np.median(ref_xy[ind[good]]-xy[good], axis=0)
    distance, ind = tree.query(xy+[dx,dy], distance_upper_bound=3.)
    good = np.isfinite(distance) & (distance < 3.)
    if good.sum() >= 4: dx, dy = np.median(ref_xy[ind[good]]-xy[good], axis=0)
    return float(dx), float(dy), int(good.sum())

reference_stars = find_stars(reference_data); shifts = []; aligned = []
for i, row in enumerate(frames.itertuples()):
    data, _ = read_frame(row.path); dx, dy, n = shift_to_reference(reference_stars, find_stars(data))
    if i == reference_index: dx, dy, n = 0., 0., len(reference_stars)
    shifts.append((dx,dy,n))
    if np.isfinite(dx): aligned.append(image_shift(data, (dy,dx), order=1, mode='constant', cval=np.nan, prefilter=False))
frames[['shift_x_pix','shift_y_pix','n_star_matches']] = shifts
display(frames[['filename','shift_x_pix','shift_y_pix','n_star_matches']])
frames.plot(x='jd_utc', y=['shift_x_pix','shift_y_pix'], marker='o', figsize=(10,3), grid=True); plt.ylabel('shift to reference, pix'); plt.show()
star_stack = np.nanmedian(np.stack(aligned), axis=0); np.save(OUTPUT/f'star_stack_{DATE}_{FILTER}.npy', star_stack)
_, sb, sr = sigma_clipped_stats(star_stack, sigma=3)
plt.figure(figsize=(11,7)); plt.imshow(star_stack, origin='lower', cmap='gray', vmin=sb-sr, vmax=sb+8*sr)
plt.title('Стек в системе неподвижных звезд: астероид должен стать следом'); plt.show()

## 4. Поиск астероида и asteroid-aligned стек

Horizons используется только как начальная координата и скорость. Если подсказка неверна или сети нет, оставьте `USE_HORIZONS=False` и впишите положение/скорость вручную после осмотра звездного стека. Настоящий объект должен повторяться на двух независимых половинах серии; одиночный дефект не считается обнаружением.

In [ ]:
mid_times = Time(frames.mid_utc.to_list(), format='isot', scale='utc')
dt_s = (mid_times-mid_times[reference_index]).to_value(u.s)
USE_HORIZONS = False
if USE_HORIZONS:
    from astroquery.jplhorizons import Horizons
    ephemeris = Horizons(id='7065', location='217', epochs=mid_times.jd).ephemerides()
    px, py = reference_wcs.world_to_pixel(SkyCoord(ephemeris['RA'], ephemeris['DEC'], unit='deg'))
    velocity_x, intercept_x = np.polyfit(dt_s, px, 1); velocity_y, intercept_y = np.polyfit(dt_s, py, 1)
    guess_x, guess_y = np.interp(0,dt_s,px), np.interp(0,dt_s,py)
else:
    # ВПИШИТЕ после визуального поиска в звездном стеке (система reference-кадра).
    guess_x, guess_y, velocity_x, velocity_y = 1000., 700., 0., 0.
print(f'initial x,y = {guess_x:.1f}, {guess_y:.1f}; v = {velocity_x:.5f}, {velocity_y:.5f} pix/s')

def moving_stack(images, shifts_xy, times_s, vx, vy):
    registered=[]
    for image, (dx,dy), dt in zip(images, shifts_xy, times_s):
        if np.isfinite(dx):
            star_aligned = image_shift(image, (dy,dx), order=1, mode='constant', cval=np.nan, prefilter=False)
            registered.append(image_shift(star_aligned, (-vy*dt,-vx*dt), order=1, mode='constant', cval=np.nan, prefilter=False))
    return np.nanmedian(np.stack(registered), axis=0)

raw_images = [read_frame(p)[0] for p in frames.path]
asteroid_stack = moving_stack(raw_images, frames[['shift_x_pix','shift_y_pix']].to_numpy(), dt_s, velocity_x, velocity_y)
np.save(OUTPUT/f'asteroid_stack_{DATE}_{FILTER}.npy', asteroid_stack)
_, ab, ar = sigma_clipped_stats(asteroid_stack, sigma=3)
plt.figure(figsize=(11,7)); plt.imshow(asteroid_stack, origin='lower', cmap='gray', vmin=ab-ar, vmax=ab+8*ar)
plt.scatter([guess_x],[guess_y],s=120,facecolors='none',edgecolors='lime',label='initial guess'); plt.legend()
plt.title('Пробный asteroid-aligned стек'); plt.show()

## 5. Положения на кадрах и внутрисерийное движение

Включайте измерения лишь когда объект убедительно виден на пробном стеке. `centroid_2dg` и формальная ошибка ниже — baseline; для публикационной точности их надо заменить взвешенным PSF/ePSF-fit и Gaia-калибровкой каждого кадра. Здесь линейная модель возвращает положение на центральную эпоху, не O−C относительно меняющейся эфемериды.

In [ ]:
MEASURE_TARGET = False; HALF_BOX = 8
def centroid_at(data, x, y):
    ix, iy = int(round(x)), int(round(y)); cut = data[iy-HALF_BOX:iy+HALF_BOX+1, ix-HALF_BOX:ix+HALF_BOX+1]
    if cut.shape != (2*HALF_BOX+1,2*HALF_BOX+1) or np.any(cut >= SATURATION): return np.nan,np.nan,np.nan
    _, bkg, rms = sigma_clipped_stats(cut, sigma=3); signal = np.maximum(cut-bkg,0)
    try: lx,ly = centroid_2dg(signal)
    except Exception: return np.nan,np.nan,np.nan
    return ix-HALF_BOX+lx, iy-HALF_BOX+ly, signal.max()/rms

def weighted_line(t,y,sigma):
    good=np.isfinite(t)&np.isfinite(y)&np.isfinite(sigma)&(sigma>0); A=np.c_[np.ones(good.sum()),t[good]]
    Aw=A/sigma[good,None]; cov=np.linalg.inv(Aw.T@Aw); beta=cov@(Aw.T@(y[good]/sigma[good]))
    return beta,cov,good,y[good]-A@beta

if MEASURE_TARGET:
    measured=[]
    for row,dt in zip(frames.itertuples(),dt_s):
        data,_=read_frame(row.path); xr=guess_x+velocity_x*dt-row.shift_x_pix; yr=guess_y+velocity_y*dt-row.shift_y_pix
        x,y,snr=centroid_at(data,xr,yr); measured.append((x+row.shift_x_pix,y+row.shift_y_pix,snr))
    result=frames.copy(); result[['x_ref_pix','y_ref_pix','peak_snr']]=measured
    sigma=np.maximum(.15,2/result.peak_snr.to_numpy()) # заменить covariance PSF-fit
    bx,covx,gx,rx=weighted_line(dt_s,result.x_ref_pix.to_numpy(),sigma); by,covy,gy,ry=weighted_line(dt_s,result.y_ref_pix.to_numpy(),sigma)
    result['model_x_ref_pix']=bx[0]+bx[1]*dt_s; result['model_y_ref_pix']=by[0]+by[1]*dt_s
    result['residual_x_pix']=result.x_ref_pix-result.model_x_ref_pix; result['residual_y_pix']=result.y_ref_pix-result.model_y_ref_pix
    result.to_csv(OUTPUT/f'positions_{DATE}_{FILTER}.csv',index=False)
    print(f't0={mid_times[reference_index].isot}; x0,y0={bx[0]:.4f},{by[0]:.4f} pix; vx,vy={bx[1]:.6f},{by[1]:.6f} pix/s')
    print(f'RMS x,y: {np.std(rx):.4f}, {np.std(ry):.4f} pix')
    fig,ax=plt.subplots(1,2,figsize=(12,3.5),sharex=True)
    ax[0].plot(dt_s/60,result.residual_x_pix,'o'); ax[0].set_ylabel('x residual, pix')
    ax[1].plot(dt_s/60,result.residual_y_pix,'o'); ax[1].set_ylabel('y residual, pix')
    for a in ax: a.axhline(0,color='k',lw=.7); a.grid(); a.set_xlabel('minutes from t0')
else: print('Сначала подтвердите объект на asteroid-aligned стеке и включите MEASURE_TARGET.')

## Критерии завершения первого этапа

- `frames_*.csv` содержит время и флаги качества, а плохие кадры исключены по понятной причине.
- Астероид виден на двух независимо собранных подстеках, а не только на полном.
- `positions_*.csv` содержит отдельные положения и остатки линейного fit. RMS сравнивается с RMS неподвижных звезд сходной яркости.
- Следующий этап: кэшировать Gaia DR3, переносить опорные звезды на эпоху, строить robust weighted plate solution и выдавать RA/Dec с полной ковариацией на центральный момент серии.

При масштабе примерно 0.49 arcsec/pix предполагаемые 8 mas — это около 0.016 pix. Поэтому на текущем этапе нельзя интерпретировать WCS-остатки как фотоцентровый сигнал.